<hr>

# ⚗️ DATA PREPROCESSING


<style>
h1 {
    text-align: left;
    color: green;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

### **dtype MAPPING**

In [ ]:

dtype_map = {
    # categorical / identifiers
    "transaction_id": "string",
    "transaction_date": "string",  # convert to datetime after loading
    "transaction_type": "string",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "property_type_code": "string",  # keep as string (categorical)
    "property_type": "string",
    "land_nature": "string",
    "land_nature_special": "string",
    "surface_type": "string",

    # lot identifiers
    "lot_1": "string",
    "lot_2": "string",
    "lot_3": "string",
    "lot_4": "string",
    "lot_5": "string",

    # numeric (use nullable Float64 for consistency)
    "property_value": "Float64",
    "building_real_surface": "Float64",
    "land_surface": "Float64",
    "lot_1_surface": "Float64",
    "lot_2_surface": "Float64",
    "lot_3_surface": "Float64",
    "lot_4_surface": "Float64",
    "lot_5_surface": "Float64",

    # integer (nullable)
    "lots_count": "Int64",
    "main_rooms_count": "Int64",
}

### **Dtype MAPPING**

In [ ]:
from pathlib import Path
import pandas as pd

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
INPUT_FILE = Path("../data/processed/CLEAN_ValeursFoncieres.csv")
COM_DEP_FILE = Path("../data/raw/COM_DEP_2026.csv")

ML_OUTPUT_FILE = Path("../data/processed/ML_ValeursFoncieres.csv")
PREPROCESS_OUTPUT_FILE = Path("../data/processed/PREPROCESS_ValeursFoncieres.csv")

CHUNK_SIZE = 50_000

# -----------------------------------------------------------------------------
# Dtypes for the DVF file
# -----------------------------------------------------------------------------
dtype_map = {
    "transaction_id": "string",
    "transaction_date": "string",
    "transaction_type": "string",
    "property_value": "float64",
    "street_number": "string",
    "btq_code": "string",
    "street_type": "string",
    "street_id": "string",
    "street_name": "string",
    "postal_code": "string",
    "com_name": "string",
    "dep_code": "string",
    "com_code": "string",
    "section_prefix": "string",
    "section": "string",
    "plot_number": "string",
    "volume_number": "string",
    "lot_1": "string",
    "lot_1_surface": "float64",
    "lot_2": "string",
    "lot_2_surface": "float64",
    "lot_3": "string",
    "lot_3_surface": "float64",
    "lot_4": "string",
    "lot_4_surface": "float64",
    "lot_5": "string",
    "lot_5_surface": "float64",
    "lots_count": "Int64",
    "property_type_code": "string",
    "property_type": "string",
    "building_real_surface": "float64",
    "main_rooms_count": "Int64",
    "land_nature": "string",
    "land_nature_special": "string",
    "land_surface": "float64",
    "surface_type": "string",
}

# -----------------------------------------------------------------------------
# Load and standardize COM_DEP_2026 mapping
# Expected output columns used here:
#   - insee_code
#   - com_type
# The function is robust to common alternative column names.
# -----------------------------------------------------------------------------
def load_com_dep_mapping(filepath: Path) -> pd.DataFrame:
    mapping = pd.read_csv(filepath, dtype=str, low_memory=False)
    mapping.columns = [c.strip() for c in mapping.columns]

    lower_map = {c.lower(): c for c in mapping.columns}

    # Try direct insee/com_type columns first
    insee_col = None
    com_type_col = None

    for candidate in ["insee_code", "code_insee", "codgeo", "codgeo", "codegeo", "code_commune_insee"]:
        if candidate in lower_map:
            insee_col = lower_map[candidate]
            break

    for candidate in ["com_type", "type_com", "typecommune", "type_commune", "typologie_commune"]:
        if candidate in lower_map:
            com_type_col = lower_map[candidate]
            break

    # If insee_code is not directly present, try building it from dep/com columns
    if insee_col is None:
        dep_col = None
        com_col = None

        for candidate in ["dep_code", "code_departement", "departement", "dep", "depcom_dep"]:
            if candidate in lower_map:
                dep_col = lower_map[candidate]
                break

        for candidate in ["com_code", "code_commune", "commune", "com", "depcom_com"]:
            if candidate in lower_map:
                com_col = lower_map[candidate]
                break

        if dep_col is None or com_col is None:
            raise ValueError(
                "Could not identify INSEE columns in COM_DEP_2026.csv. "
                "Need either an INSEE code column or department/commune code columns."
            )

        mapping["insee_code"] = (
            mapping[dep_col].astype("string").str.strip().fillna("")
            + mapping[com_col].astype("string").str.strip().str.zfill(3).fillna("")
        )
        insee_col = "insee_code"

    if com_type_col is None:
        raise ValueError(
            "Could not identify the commune type column in COM_DEP_2026.csv. "
            "Expected something like 'com_type' or 'type_commune'."
        )

    out = mapping[[insee_col, com_type_col]].copy()
    out.columns = ["insee_code", "com_type"]

    out["insee_code"] = out["insee_code"].astype("string").str.strip()
    out["com_type"] = out["com_type"].astype("string").str.strip()

    out = out.drop_duplicates(subset=["insee_code"])

    return out


# -----------------------------------------------------------------------------
# Feature engineering
# -----------------------------------------------------------------------------
def add_features(chunk: pd.DataFrame, com_dep_map: pd.DataFrame) -> pd.DataFrame:
    # insee_code = dep_code + com_code padded to 3 digits
    chunk["insee_code"] = (
        chunk["dep_code"].astype("string").str.strip().fillna("")
        + chunk["com_code"].astype("string").str.strip().str.zfill(3).fillna("")
    )
    chunk.loc[chunk["insee_code"].eq(""), "insee_code"] = pd.NA

    # merge com_type
    chunk = chunk.merge(com_dep_map, on="insee_code", how="left")

    # property_surface = building + land
    chunk["property_surface"] = (
        chunk["building_real_surface"].fillna(0)
        + chunk["land_surface"].fillna(0)
    )

    # avoid divide-by-zero
    chunk.loc[chunk["property_surface"] <= 0, "property_surface"] = pd.NA

    # value_per_m2 = property_value / property_surface
    chunk["value_per_m2"] = chunk["property_value"] / chunk["property_surface"]

    return chunk


# -----------------------------------------------------------------------------
# Columns to drop for final ML-oriented dataset
# Notes:
# - transaction_id is a unique identifier -> not useful for prediction
# - plot/section/volume/lot identifiers are mostly cadastral IDs -> high cardinality
# - street_id and raw street_name are very high-cardinality
# - value_per_m2 leaks the target because it uses property_value directly
# -----------------------------------------------------------------------------
DROP_FOR_FINAL = [
    "transaction_id",
    "section_prefix",
    "section",
    "plot_number",
    "volume_number",
    "lot_1",
    "lot_1_surface",
    "lot_2",
    "lot_2_surface",
    "lot_3",
    "lot_3_surface",
    "lot_4",
    "lot_4_surface",
    "lot_5",
    "lot_5_surface",
    "street_id",
    "street_name",
    "value_per_m2",   # target leakage
]

def drop_for_ml(chunk: pd.DataFrame) -> pd.DataFrame:
    existing = [col for col in DROP_FOR_FINAL if col in chunk.columns]
    return chunk.drop(columns=existing)


# -----------------------------------------------------------------------------
# Process and save intermediate ML file
# -----------------------------------------------------------------------------
def build_ml_file(
    input_file: Path,
    mapping_file: Path,
    output_file: Path,
    chunk_size: int = 50_000
) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)
    com_dep_map = load_com_dep_mapping(mapping_file)

    first_write = True
    total_rows = 0

    for chunk_idx, chunk in enumerate(
        pd.read_csv(
            input_file,
            chunksize=chunk_size,
            dtype=dtype_map,
            low_memory=False
        ),
        start=1
    ):
        chunk = add_features(chunk, com_dep_map)

        chunk.to_csv(
            output_file,
            mode="w" if first_write else "a",
            index=False,
            header=first_write,
            encoding="utf-8"
        )

        first_write = False
        total_rows += len(chunk)

        print(f"ML file -> chunk {chunk_idx} written: {len(chunk):,} rows | total: {total_rows:,}")

    print("\nDone! Created ML_ValeursFoncieres.csv")
    print(f"Saved file: {output_file}")
    print(f"Total rows written: {total_rows:,}")


# -----------------------------------------------------------------------------
# Process and save final preprocessing file for ML
# -----------------------------------------------------------------------------
def build_preprocessing_file(
    input_file: Path,
    output_file: Path,
    chunk_size: int = 50_000
) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)

    first_write = True
    total_rows = 0

    # dtype map for ML file includes added columns
    ml_dtype_map = dict(dtype_map)
    ml_dtype_map.update({
        "insee_code": "string",
        "com_type": "string",
        "property_surface": "float64",
        "value_per_m2": "float64",
    })

    for chunk_idx, chunk in enumerate(
        pd.read_csv(
            input_file,
            chunksize=chunk_size,
            dtype=ml_dtype_map,
            low_memory=False
        ),
        start=1
    ):
        chunk = drop_for_ml(chunk)

        chunk.to_csv(
            output_file,
            mode="w" if first_write else "a",
            index=False,
            header=first_write,
            encoding="utf-8"
        )

        first_write = False
        total_rows += len(chunk)

        print(f"PREPROCESS file -> chunk {chunk_idx} written: {len(chunk):,} rows | total: {total_rows:,}")

    print("\nDone! Created PREPROCESSING_ValeursFoncieres.csv")
    print(f"Saved file: {output_file}")
    print(f"Total rows written: {total_rows:,}")


# -----------------------------------------------------------------------------
# Run
# -----------------------------------------------------------------------------
build_ml_file(INPUT_FILE, COM_DEP_FILE, ML_OUTPUT_FILE, CHUNK_SIZE)
build_preprocessing_file(ML_OUTPUT_FILE, PREPROCESS_OUTPUT_FILE, CHUNK_SIZE)

```text
1. Data Preparation for Machine Learning
2. Train-Test SPLIT (80% Train / 20% Test) using stratify to keep same balance
3. Feature Engineering & Feauture Scaling (Cat/Num)
4. Check Data Imbalance (check if data is imbalanced and percentages)
```

<hr>

#### 🧰 INSTALLs


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: red;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

In [ ]:
# install numpy and pandas if not already installed
#!pip install numpy pandas

# install xgboost if not already installed
#!pip install xgboost

# install shap if not already installed
#!pip install shap

# install imbalanced-learn
#!pip install imbalanced-learn

<hr>

#### 📂 IMPORTs


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: red;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

from xgboost import XGBClassifier

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)

---
## **DATA PREPROCESSING**
---

### **DATA LOADING**

### **DATA PREPARATION**

```text
- ADD COLUMNS:
	insee_code = DDCCC or DDDCC (metropolean or outer-mer)
	com_type using COM_DEP_2026.csv
	property_surface = sum of the two types of surfaces
	value_per_m2 = property_value/surface
	surface_type = either "Built" or "Land"
	latitude 
	longitude
- DROP COLUMNS We don't need!
```

In [ ]:
# add columns
	insee_code = DDCCC or DDDCC (metropolean or outer-mer)
	com_type using COM_DEP_2026.csv
	property_surface = sum of the two types of surfaces
	value_per_m2 = property_value/surface
	latitude 
	longitude
- DROP COLUMNS We don't need for machine learning predict property_value based on other features!

# save file ML_ValeursFoncieres.csv	

## **Train-Test Split**

**Perform Train Test Split**
- split Valeurs Foncieres dataset into train and test sets (stratify by target variable to maintain class distribution)


**Define target (y)**
**Define features (X)**


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

X =  # features = all columns except "property_value"
y =  # target variable  = "property_value"

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y)

# save the training and testing sets to CSV files
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)


## **Feature Engineering & Feauture Scaling (Cat/Num)**

### **Feature Engineering - Categorical**

### **Feature Scaling - Numeric**